# connections-rl — analysis session: checkpoint decomposition + pass@k

One launch-and-walk-away session, GPU T4 x2 + Internet:
1. **Checkpoint curve (7B)** — structure vs semantics over GRPO training steps,
   evaluated greedy on the VAL split (test never reused). 1.5B checkpoints were
   not Hub-synced and are unrecoverable; the curve is 7B-only (stated limitation).
2. **pass@k (k=16, temp 0.9)** on the TEST split, all arms, both scales.
3. Results auto-pushed to the Hub dataset repo.

Total ~2-3h. Run all, walk away, collect.

In [ ]:
# Cell 1 — setup: repo, data, adapters, GRPO-7B checkpoints
import os
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
HF_USER = 'jacksonlukas'

!git clone https://github.com/jacksonmlukas/connections-rl.git
%cd connections-rl
!pip install -q -e . openai vllm
!git clone --depth 1 https://github.com/jacksonmlukas/gvc-local.git /kaggle/working/gvc-local
os.environ['CONNECTIONS_PUZZLES'] = '/kaggle/working/gvc-local/data/puzzles/tagged_connections.json'
!python -m connections_rl.data.build --out data/splits

from huggingface_hub import snapshot_download
for name in ['sft', 'grpo', 'sft-7b', 'grpo-7b']:
    snapshot_download(f'{HF_USER}/connections-rl-{name}', local_dir=f'adapters/{name}',
                      token=os.environ['HF_TOKEN'])
snapshot_download(f'{HF_USER}/connections-rl-grpo-7b-ckpt', local_dir='ckpts-7b',
                  token=os.environ['HF_TOKEN'], allow_patterns=['checkpoint-*/*'])
import re, pathlib
steps = sorted(int(re.search(r'\d+', d.name).group()) for d in pathlib.Path('ckpts-7b').glob('checkpoint-*'))
print('checkpoints:', steps)

In [ ]:
# Cell 2 — vLLM 7B (tp=2, fp16, eager) with base + sft + grpo + every checkpoint
import subprocess, time, urllib.request

A = '/kaggle/working/connections-rl/adapters'
ckpt_mods = ' '.join(f'ckpt-{s}=/kaggle/working/connections-rl/ckpts-7b/checkpoint-{s}' for s in steps)
cmd = (f'vllm serve Qwen/Qwen2.5-7B-Instruct --dtype half --tensor-parallel-size 2 '
       f'--enable-lora --enforce-eager --max-loras 2 '
       f'--lora-modules connections-rl-sft-7b={A}/sft-7b connections-rl-grpo-7b={A}/grpo-7b {ckpt_mods} '
       f'--max-model-len 2048 --gpu-memory-utilization 0.85')
proc = subprocess.Popen(cmd, shell=True, stdout=open('/kaggle/working/vllm.log','w'),
                        stderr=subprocess.STDOUT)
for _ in range(150):
    try:
        urllib.request.urlopen('http://localhost:8000/health'); print('vLLM 7B ready'); break
    except Exception: time.sleep(10)
else:
    raise RuntimeError('vLLM failed — check /kaggle/working/vllm.log')

In [ ]:
# Cell 3 — checkpoint decomposition curve (7B, val split, greedy)
arms = 'connections-rl-sft-7b:0,' + ','.join(f'ckpt-{s}:{s}' for s in steps)
!python -m connections_rl.eval.checkpoint_curve --arms "{arms}" \
    --puzzles data/splits/puzzles_val.json --out results-analysis/ckpt-curve-7b

In [ ]:
# Cell 4 — pass@16 on the TEST split, 7B arms
!python -m connections_rl.eval.passk --config configs/eval/qwen7b.yaml \
    --k 16 --temperature 0.9 --out results-analysis/passk-7b.json

In [ ]:
# Cell 5 — swap servers: kill 7B, serve 1.5B (single GPU) with its adapters
proc.terminate(); time.sleep(20)
!pkill -f 'vllm serve' || true
time.sleep(20)
proc2 = subprocess.Popen(
    f'vllm serve Qwen/Qwen2.5-1.5B-Instruct --dtype half --enable-lora --enforce-eager '
    f'--lora-modules connections-rl-sft={A}/sft connections-rl-grpo={A}/grpo '
    f'--max-model-len 2048 --gpu-memory-utilization 0.85',
    shell=True, stdout=open('/kaggle/working/vllm-1.5b.log','w'), stderr=subprocess.STDOUT)
for _ in range(120):
    try:
        urllib.request.urlopen('http://localhost:8000/health'); print('vLLM 1.5B ready'); break
    except Exception: time.sleep(10)
else:
    raise RuntimeError('vLLM 1.5B failed — check /kaggle/working/vllm-1.5b.log')

In [ ]:
# Cell 6 — pass@16 on the TEST split, 1.5B arms
!python -m connections_rl.eval.passk --config configs/eval/default.yaml \
    --k 16 --temperature 0.9 --out results-analysis/passk-1.5b.json

In [ ]:
# Cell 7 — durable copy to the Hub + local link
from huggingface_hub import HfApi
from IPython.display import FileLink, display
api = HfApi()
repo = f'{HF_USER}/connections-rl-results'
api.create_repo(repo, repo_type='dataset', exist_ok=True)
api.upload_folder(folder_path='results-analysis', repo_id=repo, repo_type='dataset',
                  path_in_repo='results-analysis')
print(f'pushed -> huggingface.co/datasets/{repo}/tree/main/results-analysis')
!zip -qr /kaggle/working/results-analysis.zip results-analysis
display(FileLink('/kaggle/working/results-analysis.zip'))